<a href="https://colab.research.google.com/github/gracenaomi1122/my-first-repo/blob/main/Assignment_Mini_Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ==========================================================
# Task 1 - Load and Explore the Dataset
# File: task1_load_explore.py
# ==========================================================

import pandas as pd

# ----------------------------------------------------------
# 1. Load the dataset
# ----------------------------------------------------------
try:
    df = pd.read_csv("churnguard_data.csv")
    print("Dataset loaded successfully!\n")
except FileNotFoundError:
    print("Error: 'churnguard_data.csv' not found.")
    print("Make sure the CSV file is in the same folder as this script.")
    exit()

# ----------------------------------------------------------
# 2. Print dataset shape
# ----------------------------------------------------------
print("=" * 60)
print("1. DATASET SHAPE")
print("=" * 60)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

# ----------------------------------------------------------
# 3. Print first 5 rows
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("2. FIRST FIVE ROWS")
print("=" * 60)
print(df.head())

# ----------------------------------------------------------
# 4. Print column names and data types
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("3. COLUMN INFORMATION")
print("=" * 60)
df.info()

# ----------------------------------------------------------
# 5. Print missing values
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("4. MISSING VALUES")
print("=" * 60)
print(df.isnull().sum())

# ----------------------------------------------------------
# 6. Print duplicate rows
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("5. DUPLICATE ROWS")
print("=" * 60)
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

# ----------------------------------------------------------
# 7. Print value counts of Churn column
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("6. VALUE COUNTS OF CHURN")
print("=" * 60)
print(df["Churn"].value_counts(dropna=False))

# ----------------------------------------------------------
# 8. Print unique values of Contract column
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("7. UNIQUE VALUES IN CONTRACT COLUMN")
print("=" * 60)
print(df["Contract"].unique())

# Optional: Display total number of unique contract values
print(f"\nTotal unique Contract values: {df['Contract'].nunique(dropna=False)}")

print("\n" + "=" * 60)
print("Task 1 completed successfully.")
print("=" * 60)

Dataset loaded successfully!

1. DATASET SHAPE
Rows    : 1030
Columns : 12

2. FIRST FIVE ROWS
  customerID  gender  SeniorCitizen  tenure PhoneService InternetService  \
0  CUST-0032    Male              0    21.0          YES     Fiber optic   
1  CUST-0110    Male              0    55.0          YES     Fiber optic   
2  CUST-0137  Female              1    46.0          Yes     Fiber optic   
3  CUST-0089  Female              1    63.0          Yes     Fiber optic   
4  CUST-0919  Female              0     8.0          Yes             DSl   

         Contract PaperlessBilling     PaymentMethod  MonthlyCharges  \
0  Month-to-month               No     Credit card             29.73   
1        Two year              Yes     Bank transfer           46.32   
2  Month-to-month               No    Mailed check             87.06   
3  Month-to-month              YES      Mailed check           56.97   
4  month to month               No  Electronic check           39.69   

  TotalCharges 

In [4]:
# ==========================================================
# Task 2 - Clean the Dataset
# File: task2_clean_data.py
# ==========================================================

import pandas as pd

# ----------------------------------------------------------
# 1. Load the dataset
# ----------------------------------------------------------
try:
    df = pd.read_csv("churnguard_data.csv")
    print("Dataset loaded successfully!\n")
except FileNotFoundError:
    print("Error: 'churnguard_data.csv' not found.")
    exit()

# ----------------------------------------------------------
# 2. Drop customerID column
# ----------------------------------------------------------
if "customerID" in df.columns:
    df.drop(columns=["customerID"], inplace=True)

# ----------------------------------------------------------
# 3. Remove duplicate rows
# ----------------------------------------------------------
df.drop_duplicates(inplace=True)

# ----------------------------------------------------------
# 4. Strip whitespace
# ----------------------------------------------------------
df["gender"] = df["gender"].astype(str).str.strip()
df["PaymentMethod"] = df["PaymentMethod"].astype(str).str.strip()

# ----------------------------------------------------------
# 5. Standardize casing
# ----------------------------------------------------------
for col in ["Churn", "PhoneService", "PaperlessBilling"]:
    df[col] = df[col].astype(str).str.strip().str.title()

# ----------------------------------------------------------
# 6. Fix Contract column
# ----------------------------------------------------------
df["Contract"] = (
    df["Contract"]
    .astype(str)
    .str.strip()
    .str.lower()
)

contract_mapping = {
    "month-to-month": "Month-to-month",
    "month to month": "Month-to-month",
    "monthly": "Month-to-month",

    "one year": "One year",
    "1 year": "One year",
    "one-year": "One year",

    "two year": "Two year",
    "2 year": "Two year",
    "two-year": "Two year"
}

df["Contract"] = df["Contract"].replace(contract_mapping)

# ----------------------------------------------------------
# 7. Fix InternetService column
# ----------------------------------------------------------
df["InternetService"] = (
    df["InternetService"]
    .astype(str)
    .str.strip()
    .str.lower()
)

internet_mapping = {
    "dsl": "DSL",
    "fiber optic": "Fiber optic",
    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "fiber-optic": "Fiber optic",
    "none": "No",
    "no": "No"
}

df["InternetService"] = df["InternetService"].replace(internet_mapping)

# ----------------------------------------------------------
# 8. Convert TotalCharges to numeric
# ----------------------------------------------------------
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# ----------------------------------------------------------
# Convert numeric columns
# ----------------------------------------------------------
df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce")
df["MonthlyCharges"] = pd.to_numeric(
    df["MonthlyCharges"],
    errors="coerce"
)

# ----------------------------------------------------------
# 9. Remove rows where tenure <= 0
# ----------------------------------------------------------
df = df[df["tenure"] > 0]

# ----------------------------------------------------------
# 10. Remove MonthlyCharges outliers
# ----------------------------------------------------------
df = df[
    (df["MonthlyCharges"] >= 10) &
    (df["MonthlyCharges"] <= 200)
]

# ----------------------------------------------------------
# 11. Fill missing values
# ----------------------------------------------------------

# MonthlyCharges -> mean
monthly_mean = df["MonthlyCharges"].mean()
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(monthly_mean)

# TotalCharges -> mean
total_mean = df["TotalCharges"].mean()
df["TotalCharges"] = df["TotalCharges"].fillna(total_mean)

# tenure -> median (rounded to integer)
tenure_median = round(df["tenure"].median())
df["tenure"] = df["tenure"].fillna(tenure_median)

# Convert tenure to integer
df["tenure"] = df["tenure"].astype(int)

# ----------------------------------------------------------
# 12. Print cleaned DataFrame shape
# ----------------------------------------------------------
print("=" * 60)
print("CLEANED DATASET SHAPE")
print("=" * 60)
print(df.shape)

# ----------------------------------------------------------
# 13. Print missing values
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("MISSING VALUES AFTER CLEANING")
print("=" * 60)
print(df.isnull().sum())

# ----------------------------------------------------------
# Display sample cleaned data
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("FIRST FIVE ROWS OF CLEANED DATA")
print("=" * 60)
print(df.head())

# ----------------------------------------------------------
# Save cleaned dataset (optional)
# ----------------------------------------------------------
df.to_csv("cleaned_churnguard_data.csv", index=False)

print("\nCleaning completed successfully!")
print("Cleaned dataset saved as 'cleaned_churnguard_data.csv'.")

Dataset loaded successfully!

CLEANED DATASET SHAPE
(867, 11)

MISSING VALUES AFTER CLEANING
gender              0
SeniorCitizen       0
tenure              0
PhoneService        0
InternetService     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

FIRST FIVE ROWS OF CLEANED DATA
   gender  SeniorCitizen  tenure PhoneService InternetService        Contract  \
0    Male              0      21          Yes     Fiber optic  Month-to-month   
1    Male              0      55          Yes     Fiber optic        Two year   
2  Female              1      46          Yes     Fiber optic  Month-to-month   
3  Female              1      63          Yes     Fiber optic  Month-to-month   
4  Female              0       8          Yes             DSL  Month-to-month   

  PaperlessBilling     PaymentMethod  MonthlyCharges  TotalCharges Churn  
0               No       Credit card           29.73     

In [6]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# ==========================================================
# Load Dataset
# ==========================================================
df = pd.read_csv("churnguard_data.csv")

# ==========================================================
# Cleaning (Task 2)
# ==========================================================

# Drop customerID
df.drop(columns=["customerID"], inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Strip whitespace
df["gender"] = df["gender"].str.strip()
df["PaymentMethod"] = df["PaymentMethod"].str.strip()

# Standardize casing
for col in ["PhoneService", "PaperlessBilling", "Churn"]:
    df[col] = df[col].str.strip().str.title()

# ----------------------------------------------------------
# Fix Contract
# ----------------------------------------------------------
df["Contract"] = (
    df["Contract"]
    .str.strip()
    .str.lower()
)

contract_map = {
    "month to month": "Month-to-month",
    "month-to-month": "Month-to-month",
    "monthly": "Month-to-month",

    "1 year": "One year",
    "one year": "One year",

    "2 year": "Two year",
    "two year": "Two year"
}

df["Contract"] = df["Contract"].replace(contract_map)

# ----------------------------------------------------------
# Fix InternetService
# ----------------------------------------------------------
df["InternetService"] = (
    df["InternetService"]
    .str.strip()
    .str.lower()
)

internet_map = {
    "dsl": "DSL",
    "fiber optic": "Fiber optic",
    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "none": "No",
    "no": "No"
}

df["InternetService"] = df["InternetService"].replace(internet_map)

# ----------------------------------------------------------
# Convert numeric columns
# ----------------------------------------------------------
df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce")
df["MonthlyCharges"] = pd.to_numeric(df["MonthlyCharges"], errors="coerce")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove invalid tenure
df = df[df["tenure"] > 0]

# Remove MonthlyCharges outliers
df = df[(df["MonthlyCharges"] >= 10) &
        (df["MonthlyCharges"] <= 200)]

# Fill missing values
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].mean())
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].mean())
df["tenure"] = df["tenure"].fillna(round(df["tenure"].median())).astype(int)

# ==========================================================
# Encode Target
# ==========================================================
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# ==========================================================
# One-Hot Encoding
# ==========================================================
categorical = [
    "gender",
    "PhoneService",
    "InternetService",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

df = pd.get_dummies(
    df,
    columns=categorical,
    drop_first=True
)

# ==========================================================
# Features and Target
# ==========================================================
X = df.drop("Churn", axis=1)
y = df["Churn"]

# ==========================================================
# Train-Test Split
# ==========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ==========================================================
# Standardize Numerical Features
# ==========================================================
numeric_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(
    X_train[numeric_cols]
)

X_test[numeric_cols] = scaler.transform(
    X_test[numeric_cols]
)

# ==========================================================
# Train Logistic Regression
# ==========================================================
model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

model.fit(X_train, y_train)

# ==========================================================
# Prediction
# ==========================================================
y_pred = model.predict(X_test)

# ==========================================================
# Evaluation
# ==========================================================
print("="*60)
print("Accuracy :", accuracy_score(y_test, y_pred))

print("\nClassification Report\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Stay", "Churn"]
    )
)

Accuracy : 0.6609195402298851

Classification Report

              precision    recall  f1-score   support

        Stay       0.71      0.86      0.78       119
       Churn       0.43      0.24      0.31        55

    accuracy                           0.66       174
   macro avg       0.57      0.55      0.54       174
weighted avg       0.62      0.66      0.63       174



In [9]:
# ==========================================================
# Task 4 - Customer Churn Prediction
# File: task4_predict.py
# ==========================================================

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ==========================================================
# Load Dataset
# ==========================================================

try:
    df = pd.read_csv("churnguard_data.csv")
    print("Dataset loaded successfully!\n")
except FileNotFoundError:
    print("Error: churnguard_data.csv not found.")
    exit()

# ==========================================================
# Cleaning (Task 2)
# ==========================================================

# Drop customerID
if "customerID" in df.columns:
    df.drop(columns=["customerID"], inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Strip whitespace
df["gender"] = df["gender"].astype(str).str.strip()
df["PaymentMethod"] = df["PaymentMethod"].astype(str).str.strip()

# Standardize casing
for col in ["PhoneService", "PaperlessBilling", "Churn"]:
    df[col] = df[col].astype(str).str.strip().str.title()

# ----------------------------------------------------------
# Fix Contract
# ----------------------------------------------------------

df["Contract"] = (
    df["Contract"]
      .astype(str)
      .str.strip()
      .str.lower()
)

contract_map = {
    "month-to-month": "Month-to-month",
    "month to month": "Month-to-month",
    "monthly": "Month-to-month",

    "one year": "One year",
    "1 year": "One year",

    "two year": "Two year",
    "2 year": "Two year"
}

df["Contract"] = df["Contract"].replace(contract_map)

# ----------------------------------------------------------
# Fix InternetService
# ----------------------------------------------------------

df["InternetService"] = (
    df["InternetService"]
      .astype(str)
      .str.strip()
      .str.lower()
)

internet_map = {
    "dsl": "DSL",
    "fiber optic": "Fiber optic",
    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "fiber-optic": "Fiber optic",
    "none": "No",
    "no": "No"
}

df["InternetService"] = df["InternetService"].replace(internet_map)

# ----------------------------------------------------------
# Convert numeric columns
# ----------------------------------------------------------

df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce")
df["MonthlyCharges"] = pd.to_numeric(df["MonthlyCharges"], errors="coerce")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove invalid rows
df = df[df["tenure"] > 0]

df = df[
    (df["MonthlyCharges"] >= 10) &
    (df["MonthlyCharges"] <= 200)
]

# Fill missing values
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(
    df["MonthlyCharges"].mean()
)

df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].mean()
)

df["tenure"] = (
    df["tenure"]
      .fillna(round(df["tenure"].median()))
      .astype(int)
)

# ==========================================================
# Encode Target
# ==========================================================

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# ==========================================================
# Encode Contract
# ==========================================================

contract_encoding = {
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2
}

df["Contract"] = df["Contract"].map(contract_encoding)

# ==========================================================
# Select ONLY required features
# ==========================================================

X = df[
    [
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "SeniorCitizen",
        "Contract"
    ]
]

y = df["Churn"]

# ==========================================================
# Scale Features
# ==========================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================================================
# Train Model on FULL dataset
# ==========================================================

model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

model.fit(X_scaled, y)

# ==========================================================
# User Input
# ==========================================================

print("\nEnter Customer Details\n")

tenure = int(input("Enter tenure (months): "))
monthly = float(input("Enter Monthly Charges: "))
total = float(input("Enter Total Charges: "))
senior = int(input("Senior Citizen? (1 = Yes, 0 = No): "))
contract = int(
    input(
        "Contract type (0 = Month-to-month, 1 = One year, 2 = Two year): "
    )
)

# ==========================================================
# Prepare Input
# ==========================================================

new_customer = pd.DataFrame(
    [[tenure, monthly, total, senior, contract]],
    columns=[
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "SeniorCitizen",
        "Contract"
    ]
)

new_customer_scaled = scaler.transform(new_customer)

prediction = model.predict(new_customer_scaled)

# ==========================================================
# Display Result
# ==========================================================

print("\n==============================")

if prediction[0] == 1:
    print("Prediction is 1: This customer is likely to CHURN.")
else:
    print("Prediction is 0: This customer is likely to STAY.")

print("==============================")

Dataset loaded successfully!


Enter Customer Details

Enter tenure (months): 24
Enter Monthly Charges: 65.50
Enter Total Charges: 1500.00
Senior Citizen? (1 = Yes, 0 = No): 0
Contract type (0 = Month-to-month, 1 = One year, 2 = Two year): 2

Prediction is 0: This customer is likely to STAY.
